### 08 - Inverse PINN Calibration on Real NDX Option Data

We repeat the inverse PINN experiment, but replace the synthetic Black-Scholes data with real NDX European call option market data.

r and sigma are genuinely unknown here — there is no ground truth to compare
against directly. Instead we train four separate PINNs using 50, 100, 150,
and 200 randomly selected real observations for the data-fit term, and
compare the resulting (r, sigma) estimates against each other and, later,
against a regression baseline.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from inverse_pinn_model import InversePINN
import inverse_pinn_training as ipt
from inverse_pinn_training import black_scholes_residual

device = torch.device("cpu")
dtype = torch.float64

In [2]:

# ------------------------------------------------------------
# Automatically locate NDX_call_data.csv
# ------------------------------------------------------------

from pathlib import Path

_filename = "NDX_call_data.csv"

_search_roots = [
    Path.cwd().resolve()
]

try:
    _notebook_dir = Path(__file__).resolve().parent
    _search_roots.append(_notebook_dir)
except NameError:
    pass


_candidates = []

for _root in _search_roots:

    _candidates.extend([
        _root / _filename,
        _root / "data" / _filename,
        _root.parent / _filename,
        _root.parent / "data" / _filename,
    ])


data_path = None


# First check common locations
for _candidate in _candidates:

    if _candidate.is_file():

        data_path = _candidate.resolve()
        break


# Then recursively search the current project
if data_path is None:

    for _root in _search_roots:

        try:

            _matches = list(
                _root.rglob(
                    _filename
                )
            )

        except Exception:

            _matches = []


        if _matches:

            data_path = (
                _matches[0]
                .resolve()
            )

            break


if data_path is None:

    raise FileNotFoundError(
        "\nCould not find NDX_call_data.csv.\n"
        "Make sure the CSV is included somewhere inside the downloaded project folder."
    )


print(
    "Using data file:",
    data_path
)


raw_df = pd.read_csv(data_path, index_col=0)

assert raw_df["strike_price"].nunique() == 1, "expected a single strike"
K = float(raw_df["strike_price"].iloc[0])

print("Raw shape:", raw_df.shape)
print("K:", K)
raw_df.head()

Raw shape: (239, 7)
K: 15700.0


,spot_price,current_time,strike_price,best_bid,best_offer,ticker,market_price
0,14137.230469,0.989041,15700.0,493.5,528.5,NDX,511.00
1,14270.419922,0.986301,15700.0,530.6,557.9,NDX,544.25
2,14274.240234,0.983562,15700.0,515.7,544.3,NDX,530.00
3,14365.959961,0.980822,15700.0,552.5,592.4,NDX,572.45
4,14345.179688,0.978082,15700.0,539.3,575.6,NDX,557.45


In [3]:
# current_time in the csv is time-to-maturity (tau), decreasing toward 0
# convert t = T - tau

T = float(raw_df["current_time"].max())

t_min = 0.0
t_max = T

S_min = 0.0
S_max = float(raw_df["spot_price"].max())

print("K:", K)
print("T:", T)
print("S_max:", S_max)

K: 15700.0
T: 0.989041095890411
S_max: 16573.33984375


In [4]:
# Change K, and T in synthetic data

ipt.K = K
ipt.T = T

compute_losses = ipt.compute_losses

print("inverse_pinn_training.K is now:", ipt.K)
print("inverse_pinn_training.T is now:", ipt.T)

inverse_pinn_training.K is now: 15700.0
inverse_pinn_training.T is now: 0.989041095890411


In [5]:
# rearrange original csv 
real_df = pd.DataFrame({
    "t": T - raw_df["current_time"],
    "S": raw_df["spot_price"],
    "V": raw_df["market_price"],
})

real_df = real_df.sort_values("t").reset_index(drop=True)

print("real_df shape:", real_df.shape)
real_df.head()

real_df shape: (239, 3)


,t,S,V
0,0.000000,14137.230469,511.00
1,0.002740,14270.419922,544.25
2,0.005479,14274.240234,530.00
3,0.008219,14365.959961,572.45
4,0.010959,14345.179688,557.45


In [6]:
# convert all real observations to tensors
t_all = torch.tensor(
    real_df["t"].to_numpy(), dtype=dtype, device=device
).reshape(-1, 1)

S_all = torch.tensor(
    real_df["S"].to_numpy(), dtype=dtype, device=device
).reshape(-1, 1)

V_all = torch.tensor(
    real_df["V"].to_numpy(), dtype=dtype, device=device
).reshape(-1, 1)

u_all = V_all / K

print("t_all:", t_all.shape)
print("S_all:", S_all.shape)
print("u_all:", u_all.shape)

t_all: torch.Size([239, 1])
S_all: torch.Size([239, 1])
u_all: torch.Size([239, 1])


In [7]:
data_sizes = [50, 100, 150, 200]

def sample_data_subset(n):
    """
    Randomly sample n observations (without replacement) from the real data pool
    Each call draws a fresh, independent random subset
    """
    n_pool = t_all.shape[0]
    idx = torch.randperm(n_pool)[:n]
    return t_all[idx], S_all[idx], u_all[idx]

In [8]:
# The same 500 pde collocation points for each [50, 100, 150, 200]

n_pde = 500

t_f = torch.rand(n_pde, 1, dtype=dtype, device=device) * (t_max - t_min) + t_min
S_f = torch.rand(n_pde, 1, dtype=dtype, device=device) * (S_max - S_min) + S_min

print("t_f:", t_f.shape, "range:", t_f.min().item(), t_f.max().item())
print("S_f:", S_f.shape, "range:", S_f.min().item(), S_f.max().item())

t_f: torch.Size([500, 1]) range: 0.0031954254730515463 0.987455879879183
S_f: torch.Size([500, 1]) range: 34.473656532879794 16362.748401757179


In [9]:
# 500 terminal rand points
n_terminal = 500

t_terminal = torch.full((n_terminal, 1), t_max, dtype=dtype, device=device)
S_terminal = torch.rand(n_terminal, 1, dtype=dtype, device=device) * (S_max - S_min) + S_min

print("t_terminal:", t_terminal.shape, "value:", t_terminal.unique().item())
print("S_terminal:", S_terminal.shape, "range:", S_terminal.min().item(), S_terminal.max().item())

t_terminal: torch.Size([500, 1]) value: 0.989041095890411
S_terminal: torch.Size([500, 1]) range: 75.90209163679039 16555.669879138386


In [10]:
# boundary points (left = 500, right = 500)
n_boundary = 500

t_boundary = torch.rand(n_boundary, 1, dtype=dtype, device=device) * (t_max - t_min) + t_min
S_boundary_left = torch.full((n_boundary, 1), S_min, dtype=dtype, device=device)
S_boundary_right = torch.full((n_boundary, 1), S_max, dtype=dtype, device=device)

print("t_boundary:", t_boundary.shape, "range:", t_boundary.min().item(), t_boundary.max().item())
print("S_boundary_left value:", S_boundary_left.unique().item())
print("S_boundary_right value:", S_boundary_right.unique().item())

t_boundary: torch.Size([500, 1]) range: 0.0018808748869387224 0.9877739308829808
S_boundary_left value: 0.0
S_boundary_right value: 16573.33984375


In [11]:
def build_model(seed):
    torch.manual_seed(seed)
    model = InversePINN(
        t_min=t_min,
        t_max=t_max,
        S_min=S_min,
        S_max=S_max,
        hidden_width=64,
        hidden_layers=4,
        r_initial=0.10,
        sigma_initial=0.40,
    ).to(device=device, dtype=dtype)
    return model

In [12]:
learning_rate = 1e-3
num_epochs = 30000
print_every = 1000

In [13]:
results = {}

for n_data in data_sizes:

    print(f"\nTraining with n_data = {n_data}")

    t_train, S_train, u_train = sample_data_subset(n_data)

    model = build_model(seed=n_data)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    history = {
        "epoch": [],
        "total_loss": [],
        "data_loss": [],
        "pde_loss": [],
        "terminal_loss": [],
        "boundary_loss": [],
        "r_estimate": [],
        "sigma_estimate": [],
    }

    for epoch in range(1, num_epochs + 1):

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        losses = compute_losses(
            model=model,
            t_train=t_train,
            S_train=S_train,
            u_train=u_train,
            t_f=t_f,
            S_f=S_f,
            t_terminal=t_terminal,
            S_terminal=S_terminal,
            t_boundary=t_boundary,
            S_boundary_left=S_boundary_left,
            S_boundary_right=S_boundary_right,
        )

        losses["total"].backward()

        optimizer.step()


        history["epoch"].append(
            epoch
        )

        history["total_loss"].append(
            losses["total"].item()
        )

        history["data_loss"].append(
            losses["data"].item()
        )

        history["pde_loss"].append(
            losses["pde"].item()
        )

        history["terminal_loss"].append(
            losses["terminal"].item()
        )

        history["boundary_loss"].append(
            losses["boundary"].item()
        )

        history["r_estimate"].append(
            model.r.item()
        )

        history["sigma_estimate"].append(
            model.sigma.item()
        )

        if epoch == 1 or epoch % print_every == 0:

            print(
                f"Epoch {epoch:6d} | "
                f"Total: {losses['total'].item():.6e} | "
                f"Data: {losses['data'].item():.6e} | "
                f"PDE: {losses['pde'].item():.6e} | "
                f"r: {model.r.item():.6f} | "
                f"sigma: {model.sigma.item():.6f}"
            )



    # Evaluate final PINN on ALL 239 real market observations

    model.eval()

    with torch.no_grad():

        # PINN outputs normalized price:
        # u = V / K
        u_all_prediction = model(
            t_all,
            S_all
        )

        # Convert back to original option-price scale
        V_all_prediction = (
            K * u_all_prediction
        )

        # Pricing errors on all 239 real observations
        full_data_error = (
            V_all_prediction
            -
            V_all
        )

        # MSE
        pinn_full_mse = torch.mean(
            full_data_error ** 2
        ).item()

        # RMSE
        pinn_full_rmse = np.sqrt(
            pinn_full_mse
        )

        # MAE
        pinn_full_mae = torch.mean(
            torch.abs(
                full_data_error
            )
        ).item()


    print("\nFinal evaluation on ALL real observations:")

    print(
        f"r_final         = {model.r.item():.10f}"
    )

    print(
        f"sigma_final     = {model.sigma.item():.10f}"
    )

    print(
        f"Full-data MSE   = {pinn_full_mse:.10f}"
    )

    print(
        f"Full-data RMSE  = {pinn_full_rmse:.10f}"
    )

    print(
        f"Full-data MAE   = {pinn_full_mae:.10f}"
    )



    results[n_data] = {
        "model": model,
        "history": history,
        "r_final": model.r.item(),
        "sigma_final": model.sigma.item(),
        "full_data_mse": pinn_full_mse,
        "full_data_rmse": pinn_full_rmse,
        "full_data_mae": pinn_full_mae,
    }

    torch.save(
        {
            "n_data": n_data,

            "model_state_dict": model.state_dict(),

            "r_final": model.r.item(),
            "sigma_final": model.sigma.item(),

            "full_data_mse": pinn_full_mse,
            "full_data_rmse": pinn_full_rmse,
            "full_data_mae": pinn_full_mae,

            "K": K,
            "T": T,

            "t_min": t_min,
            "t_max": t_max,

            "S_min": S_min,
            "S_max": S_max,

            "hidden_width": 64,
            "hidden_layers": 4,

            "learning_rate": learning_rate,
            "num_epochs": num_epochs,

            "history": history,
        },

        f"ndx_real_pinn_same_lr_1e-3_30000epochs_n{n_data}_run1.pt"
    )

    print(
        f"Checkpoint saved for n_data = {n_data}"
    )


Training with n_data = 50
Epoch      1 | Total: 7.464162e-01 | Data: 5.634482e-02 | PDE: 1.216327e-01 | r: 0.100095 | sigma: 0.399670
Epoch   1000 | Total: 8.323078e-04 | Data: 7.919974e-05 | PDE: 1.103797e-04 | r: 0.070869 | sigma: 0.302196
Epoch   2000 | Total: 5.425826e-04 | Data: 5.022949e-05 | PDE: 7.992479e-05 | r: 0.061760 | sigma: 0.228635
Epoch   3000 | Total: 3.260368e-04 | Data: 2.391514e-05 | PDE: 5.272207e-05 | r: 0.065161 | sigma: 0.178629
Epoch   4000 | Total: 1.683210e-04 | Data: 1.406393e-05 | PDE: 2.246796e-05 | r: 0.076486 | sigma: 0.140588
Epoch   5000 | Total: 1.495767e-04 | Data: 1.354627e-05 | PDE: 1.586329e-05 | r: 0.083644 | sigma: 0.120057
Epoch   6000 | Total: 1.442664e-04 | Data: 1.307832e-05 | PDE: 1.430497e-05 | r: 0.086514 | sigma: 0.109986
Epoch   7000 | Total: 1.403569e-04 | Data: 1.314006e-05 | PDE: 1.340959e-05 | r: 0.088613 | sigma: 0.104901
Epoch   8000 | Total: 1.520246e-04 | Data: 1.485137e-05 | PDE: 1.422331e-05 | r: 0.089458 | sigma: 0.102428
E

In [14]:
summary_df = pd.DataFrame([
    {
        "n_data": n_data,

        "r_pinn": results[n_data]["r_final"],
        "sigma_pinn": results[n_data]["sigma_final"],

        # Final training losses
        "final_data_loss": results[n_data]["history"]["data_loss"][-1],
        "final_pde_loss": results[n_data]["history"]["pde_loss"][-1],
        "final_terminal_loss": results[n_data]["history"]["terminal_loss"][-1],
        "final_boundary_loss": results[n_data]["history"]["boundary_loss"][-1],
        "final_total_loss": results[n_data]["history"]["total_loss"][-1],

        # Pricing performance on ALL 239 real observations
        "full_data_mse": results[n_data]["full_data_mse"],
        "full_data_rmse": results[n_data]["full_data_rmse"],
        "full_data_mae": results[n_data]["full_data_mae"],
    }

    for n_data in data_sizes
])


summary_df

,n_data,r_pinn,sigma_pinn,final_data_loss,final_pde_loss,final_terminal_loss,final_boundary_loss,final_total_loss,full_data_mse,full_data_rmse,full_data_mae
0,50,0.090722,0.109953,0.000009,0.000020,0.000020,0.000010,0.000116,2645.419336,51.433640,38.631715
1,100,0.089536,0.111119,0.000010,0.000016,0.000018,0.000007,0.000111,2924.154630,54.075453,39.338124
2,150,0.088030,0.118346,0.000011,0.000016,0.000014,0.000007,0.000115,2869.238136,53.565270,42.109415
3,200,0.087278,0.120121,0.000012,0.000017,0.000017,0.000008,0.000124,2791.491323,52.834566,39.191886


In [15]:
# Ablation: does the right-boundary condition (which references model.r
# directly via V_true = S_max - K*exp(-r*(T-t))) dominate the r estimate,
# independent of how much real data is used?
#
# We disable the boundary loss term entirely (lambda_boundary = 0) and
# retrain a single n_data=200 run, keeping everything else identical
# (architecture, PDE/terminal points, data size). If r_pinn moves
# noticeably toward r_tilde, the boundary condition was pulling r away
# from the data-implied value.

original_lambda_boundary = ipt.lambda_boundary
print("Original lambda_boundary:", original_lambda_boundary)

ipt.lambda_boundary = 0.0
print("Ablation lambda_boundary:", ipt.lambda_boundary)

# fresh 200-point real-data subset (independent draw, same size as before)
n_data_ablation = 200
t_train_abl, S_train_abl, u_train_abl = sample_data_subset(n_data_ablation)

model_ablation = build_model(seed=n_data_ablation)

optimizer_ablation = torch.optim.Adam(model_ablation.parameters(), lr=learning_rate)

history_ablation = {
    "epoch": [],
    "total_loss": [],
    "data_loss": [],
    "pde_loss": [],
    "terminal_loss": [],
    "boundary_loss": [],
    "r_estimate": [],
    "sigma_estimate": [],
}

print()
print(f"Starting ablation run (lambda_boundary=0, n_data={n_data_ablation})")

for epoch in range(1, num_epochs + 1):
    model_ablation.train()
    optimizer_ablation.zero_grad(set_to_none=True)

    losses = compute_losses(
        model=model_ablation,
        t_train=t_train_abl,
        S_train=S_train_abl,
        u_train=u_train_abl,
        t_f=t_f,
        S_f=S_f,
        t_terminal=t_terminal,
        S_terminal=S_terminal,
        t_boundary=t_boundary,
        S_boundary_left=S_boundary_left,
        S_boundary_right=S_boundary_right,
    )

    losses["total"].backward()
    optimizer_ablation.step()

    history_ablation["epoch"].append(epoch)
    history_ablation["total_loss"].append(losses["total"].item())
    history_ablation["data_loss"].append(losses["data"].item())
    history_ablation["pde_loss"].append(losses["pde"].item())
    history_ablation["terminal_loss"].append(losses["terminal"].item())
    history_ablation["boundary_loss"].append(losses["boundary"].item())
    history_ablation["r_estimate"].append(model_ablation.r.item())
    history_ablation["sigma_estimate"].append(model_ablation.sigma.item())

    if epoch == 1 or epoch % 5000 == 0 or epoch == num_epochs:
        print(
            f"Epoch {epoch:6d} | "
            f"Total: {losses['total'].item():.6e} | "
            f"Data: {losses['data'].item():.6e} | "
            f"PDE: {losses['pde'].item():.6e} | "
            f"r: {model_ablation.r.item():.6f} | "
            f"sigma: {model_ablation.sigma.item():.6f}"
        )

# restore the original lambda_boundary so the rest of the notebook
# (and anything re-run above) is unaffected
ipt.lambda_boundary = original_lambda_boundary
print()
print("Restored lambda_boundary:", ipt.lambda_boundary)

Original lambda_boundary: 1.0
Ablation lambda_boundary: 0.0

Starting ablation run (lambda_boundary=0, n_data=200)
Epoch      1 | Total: 6.474606e-01 | Data: 7.189704e-02 | PDE: 1.135956e-01 | r: 0.100095 | sigma: 0.399670
Epoch   5000 | Total: 1.346321e-04 | Data: 1.215321e-05 | PDE: 1.399740e-05 | r: 0.061914 | sigma: 0.125625
Epoch  10000 | Total: 6.315400e-05 | Data: 7.178208e-06 | PDE: 7.442507e-06 | r: 0.025942 | sigma: 0.175646
Epoch  15000 | Total: 5.418605e-05 | Data: 6.808542e-06 | PDE: 4.182452e-06 | r: 0.010825 | sigma: 0.186070
Epoch  20000 | Total: 6.268389e-05 | Data: 7.741410e-06 | PDE: 5.014697e-06 | r: 0.004037 | sigma: 0.189607
Epoch  25000 | Total: 5.788562e-05 | Data: 8.011726e-06 | PDE: 2.456949e-06 | r: 0.001375 | sigma: 0.189223
Epoch  30000 | Total: 5.048724e-05 | Data: 6.424772e-06 | PDE: 4.929120e-06 | r: 0.000596 | sigma: 0.187571

Restored lambda_boundary: 1.0


### Change S_max 

In [16]:
S_max_ratio = 4.0
S_max_wide = K * S_max_ratio

print("Original S_max:", S_max, "| ratio:", S_max / K)
print("New S_max:     ", S_max_wide, "| ratio:", S_max_wide / K)

S_max = S_max_wide

Original S_max: 16573.33984375 | ratio: 1.0556267416401275
New S_max:      62800.0 | ratio: 4.0


In [17]:
# PDE points
t_f = torch.rand(n_pde, 1, dtype=dtype, device=device) * (t_max - t_min) + t_min
S_f = torch.rand(n_pde, 1, dtype=dtype, device=device) * (S_max - S_min) + S_min

# terminal points
t_terminal = torch.full((n_terminal, 1), t_max, dtype=dtype, device=device)
S_terminal = torch.rand(n_terminal, 1, dtype=dtype, device=device) * (S_max - S_min) + S_min

# boundary points
t_boundary = torch.rand(n_boundary, 1, dtype=dtype, device=device) * (t_max - t_min) + t_min
S_boundary_left = torch.full((n_boundary, 1), S_min, dtype=dtype, device=device)
S_boundary_right = torch.full((n_boundary, 1), S_max, dtype=dtype, device=device)

print("New S_f range:", S_f.min().item(), S_f.max().item())
print("New S_boundary_right value:", S_boundary_right.unique().item())

New S_f range: 120.15421000980275 62605.06606062004
New S_boundary_right value: 62800.0


In [18]:
results_wide = {}

for n_data in data_sizes:

    print(f"\n{'='*60}")
    print(f"Training with n_data = {n_data} (S_max = {S_max:.1f}, ratio = {S_max_ratio})")
    print(f"{'='*60}")

    t_train, S_train, u_train = sample_data_subset(n_data)

    model = build_model(seed=n_data)   # note: build_model must use the updated S_max
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        losses = compute_losses(
            model=model,
            t_train=t_train, S_train=S_train, u_train=u_train,
            t_f=t_f, S_f=S_f,
            t_terminal=t_terminal, S_terminal=S_terminal,
            t_boundary=t_boundary,
            S_boundary_left=S_boundary_left,
            S_boundary_right=S_boundary_right,
        )
        losses["total"].backward()
        optimizer.step()

        if epoch == 1 or epoch % 5000 == 0 or epoch == num_epochs:
            print(
                f"Epoch {epoch:6d} | Total: {losses['total'].item():.6e} | "
                f"r: {model.r.item():.6f} | sigma: {model.sigma.item():.6f}"
            )

    results_wide[n_data] = {
        "r_final": model.r.item(),
        "sigma_final": model.sigma.item(),
    }


Training with n_data = 50 (S_max = 62800.0, ratio = 4.0)
Epoch      1 | Total: 1.048343e+01 | r: 0.099905 | sigma: 0.399670
Epoch   5000 | Total: 4.476951e-04 | r: 0.062062 | sigma: 0.200450
Epoch  10000 | Total: 1.069847e-03 | r: 0.052265 | sigma: 0.134426
Epoch  15000 | Total: 1.357512e-04 | r: 0.065793 | sigma: 0.103500
Epoch  20000 | Total: 1.843131e-04 | r: 0.067975 | sigma: 0.100761
Epoch  25000 | Total: 2.916157e-04 | r: 0.063253 | sigma: 0.114204
Epoch  30000 | Total: 9.360258e-05 | r: 0.058184 | sigma: 0.129089

Training with n_data = 100 (S_max = 62800.0, ratio = 4.0)
Epoch      1 | Total: 1.165617e+01 | r: 0.099905 | sigma: 0.400330
Epoch   5000 | Total: 7.234673e-04 | r: 0.060956 | sigma: 0.190452
Epoch  10000 | Total: 1.474299e-04 | r: 0.052667 | sigma: 0.131524
Epoch  15000 | Total: 1.986055e-04 | r: 0.061072 | sigma: 0.111223
Epoch  20000 | Total: 3.517200e-04 | r: 0.064314 | sigma: 0.110064
Epoch  25000 | Total: 8.935394e-05 | r: 0.061363 | sigma: 0.119349
Epoch  30000